# Bahrain Real Estate — Rent Prediction EDA

**Goal:** predict monthly `rent` (BHD) for the listings in `test.csv`.

## EDA roadmap
1. **Load & understand structure**: three scraped sources mixed together, find redundant columns
2. **Clean & standardise**: unify `offer`, parse `area_sqm`, bedrooms/bathrooms to numbers, drop duplicates
3. **Target analysis**: distribution of `rent`, outliers, log transform
4. **Univariate analysis**: each feature on its own (counts, distributions)
5. **Bivariate analysis**: each feature vs `rent` (type, governorate, area, bedrooms, furnished, ...)
6. **Missing values strategy**: decide what to impute, drop, or flag
7. **Feature ideas**: rent per sqm, location grouping, amenities count, text keywords

Every cleaning step must also be applied to `test.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('../Data/property_data.csv')
test_df = pd.read_csv('../Data/test.csv')

print('Train shape:', train_df.shape) 
print('Test shape:', test_df.shape)

Train shape: (32090, 41)
Test shape: (3566, 40)


In [56]:
train_df.head(10)

,Unnamed: 0,Property_id,property_type,offer,governorate,bedrooms,bathrooms,rent,location,city,area_sqm,listing_date,Property_id.1,URL,include_w_e,title,area,availability_date,agency,amenities,period,price_per_sqm,down_payment,payment_methods,plot_size_sqm,furnished,completion_status,location_full,latitude,longitude,agent_id,is_super_agent,broker_id,floor_plans,is_featured,is_premium,description,is_verified,is_verified.1,is_exclusive,is_exclusive.1
0,27387,7343b863-a2cb-4d85-b2dd-935ea61365ff,Apartment,Residential for Rent,"Capital Governorate, Manama",studio,1,245.0,NaN,NaN,43.0,2026-09-01T03:49:41Z,7343b863-a2cb-4d85-b2dd-935ea61365ff,https://www.propertyfinder.bh/en/plp/rent/apar...,inclusive,GREAT VALUE | FURNISHED STUDIO | PRIME LOCATION,Sanabis,NaN,NaN,"Barbecue Area, Built in Wardrobes, Central A/C...",monthly,NaN,NaN,NaN,NaN,YES,NaN,"Sanabis, Manama, Capital Governorate",26.226559,50.545910,5879.0,True,1031.0,NaN,False,False,A different furnished studio opportunity in Sa...,False,False,False,False
1,18554,f1a36ab5-1601-4127-a7a7-5095f167acca,Apartment,Rent,Capital Governorate,1,1,300.0,NaN,NaN,700 sqft / 65 sqm,NaN,f1a36ab5-1601-4127-a7a7-5095f167acca,https://www.propertyfinder.bh/en/plp/rent/apar...,Inclusive,Serviced|Free drinking water|all inclusive|Bright,Al Juffair,28 May 2025,Deccan Real Estate,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9295,bf629ae9-a3b2-4c2e-8e0e-b0ffce916fa7,Compound,Rent,Northern Governorate,5,5,1200.0,NaN,NaN,"4,306 sqft / 400 sqm",NaN,bf629ae9-a3b2-4c2e-8e0e-b0ffce916fa7,https://www.propertyfinder.bh/en/plp/rent/comp...,Exclusive,A dream home in friendly neighborhood/compound,Hamala,NaN,Aqaratcom For Real Estate,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28118,26707746-825c-4d65-b63c-dfaa69421365,Apartment,Residential for Rent,Capital Governorate,3,3,600.0,NaN,NaN,160.0,2026-08-24T07:11:37Z,26707746-825c-4d65-b63c-dfaa69421365,https://www.propertyfinder.bh/en/plp/rent/apar...,inclusive,Spacious 3 Bedroom Apartment for Rent in Seef,Seef,NaN,NaN,"Balcony, Central A/C, Covered Parking, Kitchen...",monthly,NaN,NaN,NaN,NaN,YES,NaN,"Seef, Capital Governorate",26.240000,50.536999,7237.0,False,15.0,NaN,True,False,Spacious 3 Bedroom Apartment for Rent in Seef ...,False,False,False,False
4,689,dbc18ce8-1481-480c-9b66-a42653038408,Apartment,for rent,Muharraq Governorate,1,2,450.0,Al Marsa Floating City,Amwaj Islands,80.0,2024-11-15,dbc18ce8-1481-480c-9b66-a42653038408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6444,4573ed83-c6a3-4fd8-8c48-2d85bf4042d0,Apartment,for rent,Muharraq Governorate,2,2,180.0,NaN,Busaiteen,120.0,2023-09-21,4573ed83-c6a3-4fd8-8c48-2d85bf4042d0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,3240,7048fd82-1af7-4556-bd84-39bab8283061,Apartment,for rent,Muharraq Governorate,1,2,350.0,The Treasure,Dilmunia Island,91.0,2024-05-13,7048fd82-1af7-4556-bd84-39bab8283061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,23912,02463af5-1ab9-4537-95e3-9bf6d820445b,Villa,Residential for Rent,Northern Governorate,5,5,900.0,NaN,NaN,500.0,2026-09-19T14:52:18Z,02463af5-1ab9-4537-95e3-9bf6d820445b,https://www.propertyfinder.bh/en/plp/rent/vill...,notSelected,Premium 5-BR Executive Villa with Private Garden,Janabiya,NaN,NaN,"Maids Room, Central A/C, Balcony, Private Gard...",monthly,NaN,NaN,NaN,500.0,PARTLY,NaN,"Janabiya, Northern Governorate",26.185944,50.467113,5089.0,True,850.0,NaN,True,False,"Discover the perfect blend of space, luxury, a...",False,False,False,False
8,19226,8e137aa1-0967-43ba-a584-b6faeb9d9a31,Apartment,Rent,Capital Governorate,1,2,650.0,NaN,NaN,"1,292 sqft / 120 sqm",NaN,8e137aa1-0967-43ba-a584-b6faeb9d9a31,https://www.propertyfinder.bh/en/plp/rent/apar...,Exclusive,Amazing DUPLEX 1 BR

In [57]:
train_df.sample(5, random_state=42)

,Unnamed: 0,Property_id,property_type,offer,governorate,bedrooms,bathrooms,rent,location,city,area_sqm,listing_date,Property_id.1,URL,include_w_e,title,area,availability_date,agency,amenities,period,price_per_sqm,down_payment,payment_methods,plot_size_sqm,furnished,completion_status,location_full,latitude,longitude,agent_id,is_super_agent,broker_id,floor_plans,is_featured,is_premium,description,is_verified,is_verified.1,is_exclusive,is_exclusive.1
23770,21946,d3dfdcbd-a929-476d-b4d3-c23b8e1dfe13,Villa,Residential for Rent,Northern Governorate,4,4,1100.0,NaN,NaN,400.0,2026-09-27T20:04:17Z,d3dfdcbd-a929-476d-b4d3-c23b8e1dfe13,https://www.propertyfinder.bh/en/plp/rent/vill...,inclusive,FRESH / GOOD FACILITIES / NEAR BSB / QUIET AREA,Al Jasra,NaN,NaN,"Covered Parking, Balcony, Barbecue Area, Built...",monthly,NaN,NaN,NaN,400.0,YES,NaN,"Al Jasra, Northern Governorate",26.166685,50.450787,5129.0,False,904.0,NaN,True,False,"Eva Realty presents this Double Story , Stunni...",False,False,False,False
27980,10566,2bc446ab-3ee6-4e7f-aca9-8f79c302e0aa,Apartment,Rent,Capital Governorate,2,2,480.0,NaN,NaN,"1,345 sqft / 125 sqm",NaN,2bc446ab-3ee6-4e7f-aca9-8f79c302e0aa,https://www.propertyfinder.bh/en/plp/rent/apar...,Inclusive,BRAND NEW/ PRIME LOCATION/ GREAT AMENITIES/WIFI,Seef,27 Aug 2025,Diyarna Properties,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9248,7461,e0f0212c-3b90-47f9-9368-f863f7ac9670,Apartment,for rent,Muharraq Governorate,2,2,220.0,NaN,Hidd,120.0,2023-06-11,e0f0212c-3b90-47f9-9368-f863f7ac9670,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25160,19373,37b7be90-4d1c-4169-9519-53b299ec92d1,Apartment,Rent,Muharraq Governorate,3+ Maid,4,900.0,NaN,NaN,"2,034 sqft / 189 sqm",NaN,37b7be90-4d1c-4169-9519-53b299ec92d1,https://www.propertyfinder.bh/en/plp/rent/apar...,Exclusive,Spacious | Balcony | Maids room | Sea view,"The Treasure, Dilmunia Island",18 Mar 2025,Kensington Real Estate,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14928,7126,72e564c5-79ec-4b2a-b225-7b9e1aa84777,Apartment,for rent,Capital Governorate,2,3,550.0,Mahooz,Manama,80.0,2023-07-23,72e564c5-79ec-4b2a-b225-7b9e1aa84777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Notes - first look at the data
- **offer**: all values seem to mean "for rent", so it probably doesn't help predict rent (to be confirmed in the next cell).
- **area_sqm**: some values contain both sqft and sqm (e.g. `"700 sqft / 65 sqm"`). Only the sqm number should be kept.
- **bedrooms**: not all values are integers. The non-integer values need to be converted to integers.

In [59]:
train_df['offer'].value_counts()

offer
Residential for Rent    14358
Rent                     9526
for rent                 8206
Name: count, dtype: int64

- **offer**: 3 values ("Residential for Rent", "Rent", "for rent") that all mean the same thing. They come from 3 different scrapes of the data, and each scrape fills different columns. Not useful for predicting rent directly, but useful for knowing where a row came from.

In [62]:
train_df['bedrooms'].value_counts(dropna=False)

bedrooms
2               12846
1                6152
3                5756
4                2797
studio           1480
5                 745
4+ Maid           744
3+ Maid           671
2+ Maid           258
5+ Maid           225
6                 117
7+                 71
0                  59
1+ Maid            46
7                  43
6+ Maid            39
NaN                12
studio+ Maid        8
10                  7
7+ Maid             7
7++ Maid            7
Name: count, dtype: int64

- **bedrooms** — conversion plan:
  - `studio` → 0
  - `"N+ Maid"` → N bedrooms + a separate column `has_maid_room = 1` (keeps the maid-room signal separate from bedroom count)
  - `"7+"` → 7
  - 12 rows with no bedrooms in train → delete (too few to matter).
  - ⚠️ Test rows with no bedrooms can't be deleted → fill them later (Step 6).

In [63]:
train_df['bathrooms'].value_counts(dropna=False)

bathrooms
2       13063
3        7592
1        4098
4        3804
5        2436
6         728
7         209
7+        133
10         20
NaN         4
none        3
Name: count, dtype: int64

- **bathrooms** — conversion plan:
  - `"7+"` → 7
  - `"none"` → treat as missing (same as NaN)
  - Missing rows in train → delete.
  - ⚠️ Test rows with missing bathrooms can't be deleted → fill them later (Step 6).

In [65]:
area_as_number = pd.to_numeric(train_df['area_sqm'], errors='coerce')
area_is_text = area_as_number.isna() & train_df['area_sqm'].notna()

print('Rows where area_sqm is not a plain number:', area_is_text.sum())
train_df.loc[area_is_text, 'area_sqm'].head(10)

Rows where area_sqm is not a plain number: 9526


1           700 sqft / 65 sqm
2        4,306 sqft / 400 sqm
8        1,292 sqft / 120 sqm
13       1,475 sqft / 137 sqm
16    10,764 sqft / 1,000 sqm
17       1,184 sqft / 110 sqm
24       3,229 sqft / 300 sqm
29          915 sqft / 85 sqm
32       1,076 sqft / 100 sqm
37       1,938 sqft / 180 sqm
Name: area_sqm, dtype: str

In [67]:
train_df.loc[area_is_text, 'offer'].value_counts()

offer
Rent    9526
Name: count, dtype: int64

- **area_sqm** — 9,526 rows are text in the format `"X sqft / Y sqm"` (all from the "Rent" scrape). All other rows are plain numbers.
  - Plan: extract the number before "sqm" and convert the column to numbers.

In [68]:
missing_percent = train_df.isna().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False).round(1)
missing_percent

price_per_sqm        100.0
completion_status    100.0
floor_plans          100.0
down_payment         100.0
payment_methods       99.3
plot_size_sqm         92.3
location              91.0
city                  74.4
availability_date     71.7
agency                70.3
furnished             55.3
is_premium            55.3
description           55.3
is_verified           55.3
is_verified.1         55.3
is_exclusive          55.3
is_exclusive.1        55.3
broker_id             55.3
is_featured           55.3
latitude              55.3
longitude             55.3
location_full         55.3
is_super_agent        55.3
agent_id              55.3
period                55.3
listing_date          29.7
amenities             26.6
include_w_e           25.6
URL                   25.6
title                 25.6
area                  25.6
bedrooms               0.0
bathrooms              0.0
rent                   0.0
property_type          0.0
offer                  0.0
governorate            0.0
P

- **Missing values**
  - 100% empty → delete: `price_per_sqm`, `completion_status`, `floor_plans`, `down_payment`
  - Almost empty → decide in Step 6:
    - `payment_methods` (~99% missing) — probably too few values to use
    - `plot_size_sqm` (~92% missing) — may be useful: likely only filled for villas/houses

In [69]:
for column_name in train_df.columns:
    number_of_unique_values = train_df[column_name].nunique()
    if number_of_unique_values == 1:
        only_value = train_df[column_name].dropna().iloc[0]
        print(column_name, '→ only value:', only_value)

period → only value: monthly
is_verified → only value: False
is_verified.1 → only value: False
is_exclusive → only value: False
is_exclusive.1 → only value: False


- **Single-value columns → delete**
  - `period`: always "monthly" (rent is monthly by definition)
  - `is_verified`, `is_exclusive` (+ their `.1` copies): always False; blanks are just the scrapes that didn't record them

In [71]:
column_pairs = [
    ('Property_id', 'Property_id.1'),
    ('is_verified', 'is_verified.1'),
    ('is_exclusive', 'is_exclusive.1'),
]

for original_column, copy_column in column_pairs:
    are_identical = train_df[original_column].equals(train_df[copy_column])
    print(original_column, 'vs', copy_column, '→ identical:', are_identical)

Property_id vs Property_id.1 → identical: True
is_verified vs is_verified.1 → identical: True
is_exclusive vs is_exclusive.1 → identical: True


- **Duplicate columns → delete**: `Property_id.1`, `is_verified.1`, `is_exclusive.1` (exact copies)

In [73]:
columns_to_delete = [
    # 100% empty
    'price_per_sqm', 'completion_status', 'floor_plans', 'down_payment',
    # only one value
    'period', 'is_verified', 'is_exclusive',
    # exact copies
    'Property_id.1', 'is_verified.1', 'is_exclusive.1',
]

train_df = train_df.drop(columns=columns_to_delete)
test_df = test_df.drop(columns=columns_to_delete)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

Train shape: (32090, 31)
Test shape: (3566, 30)


## Step 2 — Cleaning
Apply every cleaning step to **both** train and test, so the model sees the same format in both.

In [74]:
def convert_bedrooms_to_number(bedrooms_value):
    if pd.isna(bedrooms_value):
        return np.nan
    bedrooms_text = str(bedrooms_value).lower()
    if bedrooms_text.startswith('studio'):
        return 0
    number_part = bedrooms_text.split('+')[0]
    return int(number_part)


for dataframe in [train_df, test_df]:
    dataframe['has_maid_room'] = dataframe['bedrooms'].str.contains('Maid', na=False).astype(int)
    dataframe['bedrooms'] = dataframe['bedrooms'].apply(convert_bedrooms_to_number)

train_df['bedrooms'].value_counts(dropna=False).sort_index()

bedrooms
0.0      1547
1.0      6198
2.0     13104
3.0      6427
4.0      3541
5.0       970
6.0       156
7.0       128
10.0        7
NaN        12
Name: count, dtype: int64

In [75]:
def convert_bathrooms_to_number(bathrooms_value):
    if pd.isna(bathrooms_value):
        return np.nan
    bathrooms_text = str(bathrooms_value).lower()
    if bathrooms_text == 'none':
        return np.nan
    number_part = bathrooms_text.split('+')[0]
    return int(number_part)


for dataframe in [train_df, test_df]:
    dataframe['bathrooms'] = dataframe['bathrooms'].apply(convert_bathrooms_to_number)

train_df['bathrooms'].value_counts(dropna=False).sort_index()

bathrooms
1.0      4098
2.0     13063
3.0      7592
4.0      3804
5.0      2436
6.0       728
7.0       342
10.0       20
NaN         7
Name: count, dtype: int64

In [76]:
rows_before = len(train_df)

train_df = train_df.dropna(subset=['bedrooms', 'bathrooms'])

rows_after = len(train_df)
print('Rows deleted:', rows_before - rows_after)
print('Train shape:', train_df.shape)

print('Test rows with missing bedrooms:', test_df['bedrooms'].isna().sum())
print('Test rows with missing bathrooms:', test_df['bathrooms'].isna().sum())

Rows deleted: 15
Train shape: (32075, 32)
Test rows with missing bedrooms: 2
Test rows with missing bathrooms: 2


- Deleted train rows with missing `bedrooms` or `bathrooms` (15 rows).
- Test still has 2 missing bedrooms and 2 missing bathrooms will be filled later on.

In [77]:
def convert_area_to_sqm(area_value):
    if pd.isna(area_value):
        return np.nan
    area_text = str(area_value)
    sqm_part = area_text.split('/')[-1]
    sqm_part = sqm_part.replace('sqm', '').replace(',', '').strip()
    return float(sqm_part)


for dataframe in [train_df, test_df]:
    dataframe['area_sqm'] = dataframe['area_sqm'].apply(convert_area_to_sqm)

train_df['area_sqm'].describe().round(1)

count    32075.0
mean       173.9
std        147.5
min          1.0
25%         98.0
50%        130.0
75%        182.0
max       5000.0
Name: area_sqm, dtype: float64

- **area_sqm** converted to numbers. No missing values.
  - min = 1 sqm → impossible, likely a typo/placeholder
  - max = 5,000 sqm → possibly real (whole building / large villa)

In [83]:
train_df['governorate'].value_counts(dropna=False)

governorate
Capital Governorate                        12849
Muharraq Governorate                        6222
Northern Governorate                        6067
Capital Governorate, Manama                 2247
Muharraq Governorate, Amwaj Islands         1538
Central Governorate                         1109
Muharraq Governorate, Diyar Al Muharraq      691
Southern Governorate                         540
Southern Governorate, Riffa                  356
Muharraq Governorate, Dilmunia Island        290
Capital Governorate, Hoora                   103
Northern Governorate, Bu Quwah                33
Southern Governorate, Durrat Al Bahrain       18
Southern Governorate, Zallaq                  12
Name: count, dtype: int64

In [84]:
print('Central rows in train:', (train_df['governorate'] == 'Central Governorate').sum())
print('Central rows in test:', (test_df['governorate'] == 'Central Governorate').sum())

Central rows in train: 1109
Central rows in test: 135


- **governorate**: 4 current governorates + "Central Governorate" (abolished in 2014).
  - Decision: keep "Central" as its own category — deleting would lose ~1,200 rows and test also contains Central rows; remapping by city is error-prone. The `area` column gives more precise location anyway.

In [85]:
for dataframe in [train_df, test_df]:
    dataframe['governorate'] = dataframe['governorate'].str.split(',').str[0]

train_df['governorate'].value_counts(dropna=False)

governorate
Capital Governorate     15199
Muharraq Governorate     8741
Northern Governorate     6100
Central Governorate      1109
Southern Governorate      926
Name: count, dtype: int64

In [86]:
location_columns = ['area', 'city', 'location']

filled_counts = train_df[location_columns].notna().groupby(train_df['offer']).sum()
filled_counts

,area,city,location
offer,,,
Rent,9526,0,0
Residential for Rent,14343,0,0
for rent,0,8206,2899


In [87]:
for column_name in location_columns:
    print(column_name, '→', train_df[column_name].dropna().unique()[:10])

area → <StringArray>
[                  'Sanabis',                'Al Juffair',
                    'Hamala',                      'Seef',
                  'Janabiya',                 'Busaiteen',
                      'Hidd', 'The Lagoon, Amwaj Islands',
                'Bu Kowarah',            'Mahooz, Manama']
Length: 10, dtype: str
city → <StringArray>
[    'Amwaj Islands',         'Busaiteen',   'Dilmunia Island',
          'Janabiya',       'Reef Island',              'Saar',
        'Al Juffair',            'Hamala', 'Diyar Al Muharraq',
              'Seef']
Length: 10, dtype: str
location → <StringArray>
['Al Marsa Floating City',           'The Treasure',             'The Lagoon',
            'Tala Island',      'Marassi Boulevard',            'Um Al Hasam',
             'Al Burhama',                'Sanabis',           'Amwaj Avenue',
                 'alnaim']
Length: 10, dtype: str


- **area / city**: same information (neighbourhood), stored in different columns by different scrapes → combine into `area`.
- **location**: a smaller place inside a neighbourhood (project/building). Only filled for some "for rent" rows.

In [88]:
for dataframe in [train_df, test_df]:
    dataframe['area'] = dataframe['area'].fillna(dataframe['city'])

print('Missing area in train:', train_df['area'].isna().sum())
print('Missing area in test:', test_df['area'].isna().sum())

Missing area in train: 0
Missing area in test: 0


In [89]:
train_df['area'].nunique()

208

In [90]:
area_has_comma = train_df['area'].str.contains(',')

print('Rows with a comma in area:', area_has_comma.sum())
train_df.loc[area_has_comma, 'area'].value_counts().head(15)

Rows with a comma in area: 3433


area
Amwaj Avenue, Amwaj Islands              404
Mahooz, Manama                           317
Adliya, Manama                           281
The Lagoon, Amwaj Islands                199
Um Al Hasam, Manama                      187
Zinj, Manama                             163
Sanabis, Manama                          158
Marassi Al Bahrain, Diyar Al Muharraq    147
Amwaj Marina, Amwaj Islands              127
Al Burhama, Manama                        96
Bahrain Financial Harbour, Manama         96
Amwaj Beachfront, Amwaj Islands           93
Segaya, Manama                            87
Tala Island, Amwaj Islands                87
Amwaj Homes, Amwaj Islands                79
Name: count, dtype: int64

In [91]:
for dataframe in [train_df, test_df]:
    area_has_comma = dataframe['area'].str.contains(',', na=False)
    area_parts = dataframe.loc[area_has_comma, 'area'].str.split(',')

    dataframe.loc[area_has_comma, 'location'] = area_parts.str[0].str.strip()
    dataframe.loc[area_has_comma, 'area'] = area_parts.str[-1].str.strip()

print('Unique areas now:', train_df['area'].nunique())
print('Rows with a comma left:', train_df['area'].str.contains(',').sum())

Unique areas now: 142
Rows with a comma left: 0


In [92]:
train_df = train_df.drop(columns=['city'])
test_df = test_df.drop(columns=['city'])

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

Train shape: (32075, 31)
Test shape: (3566, 30)


- **area**: filled from `city` for the "for rent" rows → no missing values.
- Values like `"The Lagoon, Amwaj Islands"` split: neighbourhood → `area`, sub-area → `location`.
- Unique neighbourhoods: 208 → 142 after the split.
- `city` deleted (now fully inside `area`).

In [93]:
train_df['include_w_e'].value_counts(dropna=False)

include_w_e
inclusive      9442
NaN            8206
Inclusive      5834
notSelected    4800
Exclusive      3692
exclusive       101
Name: count, dtype: int64